# Kubernetes Failure Prediction: Model Evaluation

This notebook evaluates the models we trained in the previous notebook and analyzes their performance.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import plotly.express as px
import plotly.graph_objects as go

# Set up paths to access parent directory modules
import sys
sys.path.append('..')

# Import custom modules
from model_evaluator import evaluate_model, evaluate_time_series_model
from visualizer import (plot_confusion_matrix, plot_metrics_over_time, 
                        plot_anomaly_detection, create_classification_performance_matrix,
                        create_time_series_performance_matrix)

## Load Models and Test Data

First, we'll load the models we trained and the test data we saved.

In [ ]:
# Load the models
try:
    rf_model = joblib.load('../models/random_forest_model.pkl')
    if_model = joblib.load('../models/isolation_forest_model.pkl')
    try:
        ts_model = joblib.load('../models/time_series_model.pkl')
        has_ts_model = True
    except FileNotFoundError:
        print("Time series model not found. Will skip time series evaluation.")
        has_ts_model = False
    print("Models loaded successfully.")
except FileNotFoundError:
    print("Models not found. Please run the model_training notebook first.")
    raise FileNotFoundError("Required model files are missing.")

# Load the test data
try:
    train_test_data = joblib.load('../data/train_test_data.pkl')
    X_train = train_test_data['X_train']
    X_test = train_test_data['X_test']
    y_train = train_test_data['y_train']
    y_test = train_test_data['y_test']
    print(f"Test data loaded with {X_test.shape[0]} samples.")
except FileNotFoundError:
    print("Test data not found. Please run the model_training notebook first.")
    raise FileNotFoundError("Required test data is missing.")

## Evaluate Random Forest Model

Let's evaluate the performance of our Random Forest model on the test data.

In [ ]:
# Evaluate the Random Forest model
print("Evaluating Random Forest model...")
rf_evaluation = evaluate_model(
    rf_model, 
    X_test, 
    y_test,
    model_type='random_forest'
)

# Display key metrics
print("\nRandom Forest Performance Metrics:")
for metric, value in rf_evaluation.items():
    if metric in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
        print(f"{metric.capitalize()}: {value:.4f}")

## Visualize Random Forest Results

Now let's create visualizations to better understand the model's performance.

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(8, 6))
confusion_fig = plot_confusion_matrix(y_test, rf_evaluation['predictions'])
plt.show()

# Create and display the classification performance matrix
display_metrics = {}
for key, value in rf_evaluation.items():
    if key in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
        display_metrics[key] = value

perf_fig = create_classification_performance_matrix(
    display_metrics, 
    model_name="Random Forest Classifier"
)
perf_fig.show()

## Evaluate Isolation Forest Model

Next, let's evaluate the Isolation Forest model for anomaly detection.

In [ ]:
# Evaluate the Isolation Forest model
print("Evaluating Isolation Forest model...")
if_evaluation = evaluate_model(
    if_model, 
    X_test, 
    y_test,
    model_type='isolation_forest'
)

# Display key metrics
print("\nIsolation Forest Performance Metrics:")
for metric, value in if_evaluation.items():
    if metric in ['accuracy', 'precision', 'recall', 'f1']:
        print(f"{metric.capitalize()}: {value:.4f}")

## Visualize Isolation Forest Results

Let's visualize the results of our anomaly detection model.

In [ ]:
# Visualize anomaly detection with PCA for dimensionality reduction
anomaly_fig = plot_anomaly_detection(X_test, if_evaluation['predictions'], n_components=2)
anomaly_fig.show()

# Create and display the performance matrix for Isolation Forest
display_metrics = {}
for key, value in if_evaluation.items():
    if key in ['accuracy', 'precision', 'recall', 'f1']:
        display_metrics[key] = value

perf_fig = create_classification_performance_matrix(
    display_metrics, 
    model_name="Isolation Forest Anomaly Detection"
)
perf_fig.show()

## Compare Model Performance

Let's compare the performance of our classification and anomaly detection models.

In [ ]:
# Create a comparison dataframe
model_names = ['Random Forest', 'Isolation Forest']
metrics = ['accuracy', 'precision', 'recall', 'f1']

comparison_data = []
for metric in metrics:
    comparison_data.append({
        'Metric': metric.capitalize(),
        'Random Forest': rf_evaluation.get(metric, 0),
        'Isolation Forest': if_evaluation.get(metric, 0)
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df

In [ ]:
# Create a bar chart for visual comparison
comparison_melted = pd.melt(comparison_df, id_vars=['Metric'], var_name='Model', value_name='Score')

fig = px.bar(comparison_melted, x='Metric', y='Score', color='Model', barmode='group',
             title='Model Performance Comparison', height=500,
             labels={'Score': 'Performance Score (0-1)', 'Metric': 'Performance Metric'},
             text_auto='.3f')
fig.update_layout(yaxis_range=[0, 1])
fig.show()

## Evaluate Time Series Model (if available)

If we have a time series model, let's evaluate its forecasting performance.

In [ ]:
if has_ts_model:
    print("Evaluating Time Series model...")
    
    # Load original data to get actual values for comparison
    try:
        original_data = pd.read_csv('../data/preprocessed_kubernetes_data.csv')
    except FileNotFoundError:
        # If original data not available, generate new data
        from data_generator import generate_kubernetes_data
        original_data = generate_kubernetes_data(5000, 0.1, 30)
    
    # Extract feature and forecasts from the model dictionary
    feature = ts_model.get('feature')
    forecast = ts_model.get('forecast')
    fitted_model = ts_model.get('model')
    
    if feature and forecast is not None and not isinstance(forecast, type(None)):
        # Get actual values for comparison
        actual_values = original_data[feature].values
        
        # Take a portion for testing
        n_test = min(len(forecast), len(actual_values) // 3)
        if n_test > 0:
            test_actual = actual_values[-n_test:]
            test_pred = ts_model.get('forecasted_history', forecast[:n_test])
            
            # Evaluate the time series model
            ts_metrics = evaluate_time_series_model(test_actual, test_pred, ts_model.get('model_info'))
            
            # Print time series metrics
            print(f"\nTime Series Model ({feature}) Performance Metrics:")
            for metric, value in ts_metrics.items():
                if metric not in ['metrics_available', 'model_info']:
                    print(f"{metric.upper()}: {value:.4f}")
            
            # Plot the time series forecast vs actual values
            fig = plot_metrics_over_time(original_data, ts_model)
            fig.show()
            
            # Create and display the time series performance matrix
            ts_perf_fig = create_time_series_performance_matrix(ts_metrics, model_name=f"{feature} ARIMA Model")
            ts_perf_fig.show()
        else:
            print("Not enough data for time series evaluation.")
    else:
        print("Invalid time series model or missing forecast data.")
else:
    print("No time series model available for evaluation.")

## Conclusion

In this notebook, we've evaluated the performance of our machine learning models for Kubernetes failure prediction:

1. **Random Forest** showed strong performance in classification metrics, particularly in [insert specific strengths].

2. **Isolation Forest** performed [compared to Random Forest] in anomaly detection, with [strengths and weaknesses].

3. **Time Series Forecasting** with ARIMA demonstrated the ability to predict future metric values with [level of accuracy] accuracy.

Based on these evaluations, we can conclude that [insert final insights about model performance and their applicability to real-world Kubernetes monitoring scenarios].

The next steps would be to:
1. Deploy these models in a Kubernetes environment
2. Set up real-time monitoring and alerting
3. Implement a feedback loop for continuous model improvement